In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
import yfinance as yf

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
import os
import random

def fijar_semillas(semilla=42):
    # 1. Fijar semilla de Python
    os.environ['PYTHONHASHSEED'] = str(semilla)
    random.seed(semilla)
    
    # 2. Fijar semilla de NumPy
    np.random.seed(semilla)
    
    # 3. Fijar semilla de TensorFlow/Keras
    tf.random.set_seed(semilla)
    
    print(f"[*] Semillas fijadas a {semilla} para asegurar reproducibilidad.")

# Llamar a la función antes de crear ningún modelo ni dividir datos
fijar_semillas(42)

[*] Semillas fijadas a 42 para asegurar reproducibilidad.


In [3]:
# =====================================================================
# 1. DESCARGA Y PREPARACIÓN DE DATOS: PORTFOLIO DOLLAR BARS
# =====================================================================
print("Descargando datos de Yahoo Finance (Precios y Volumen)...")
start_date = '1960-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 'GE', 
                   'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 'MO', 'MRK', 
                   'MSI', 'PG', 'XOM']

# Descargamos todos los datos (OHLCV)
datos_yahoo = yf.download(tickers_validos, start=start_date, auto_adjust=True, progress=False)

# Separamos Precios de Cierre y Volúmenes, eliminando activos con datos faltantes
precios_close = datos_yahoo['Close'].dropna(axis=1)
volumenes = datos_yahoo['Volume'].dropna(axis=1)

print(f"Datos cronológicos originales (Time Bars): {precios_close.shape}")

# ---------------------------------------------------------------------
# A) Cálculo del Dollar Volume del Portfolio
# ---------------------------------------------------------------------
# Multiplicamos precio por volumen para cada activo (Dólares negociados por empresa y día)
dollar_volume_individual = precios_close * volumenes

# Sumamos horizontalmente para obtener los Dólares negociados por TODAS las 23 empresas ese día
portfolio_dollar_volume = dollar_volume_individual.sum(axis=1)


# ---------------------------------------------------------------------
# B) Función de Muestreo de Dollar Bars
# ---------------------------------------------------------------------
def sample_portfolio_dollar_bars(df_precios, serie_dollar_vol, threshold):
    """
    Recorre la serie cronológica acumulando dólares negociados.
    Cuando el acumulado supera el 'threshold' (umbral), guarda la fecha,
    cierra la barra y resetea el contador.
    """
    fechas_barras = []
    vol_acumulado = 0.0
    
    # Extraemos fechas y volúmenes para iterar rápidamente
    fechas = serie_dollar_vol.index
    volumenes_diarios = serie_dollar_vol.values
    
    for fecha, vol in zip(fechas, volumenes_diarios):
        # Ignoramos NaNs que puedan surgir en datos muy antiguos
        if np.isnan(vol):
            continue
            
        vol_acumulado += vol
        
        # ¿Hemos superado la "Triple Barrera" de volumen de dinero?
        if vol_acumulado >= threshold:
            fechas_barras.append(fecha)
            vol_acumulado = 0.0  # Reset de la barra
            
    # Filtramos la matriz original de 23 activos usando solo las fechas donde se cerró una barra
    return df_precios.loc[fechas_barras]

# ---------------------------------------------------------------------
# C) Generación de la nueva matriz indexada por Información
# ---------------------------------------------------------------------
# Definir el umbral (Threshold). 
# Prado recomienda usar una fracción del volumen total, o un múltiplo de la media diaria.
# Por ejemplo: queremos que cada barra contenga la cantidad de dólares que, EN MEDIA, se negocian en 5 días.
umbral_dolares = portfolio_dollar_volume.mean() * 5 

precios_dollar_bars = sample_portfolio_dollar_bars(precios_close, portfolio_dollar_volume, umbral_dolares)

print(f"Nueva matriz tras compresión (Portfolio Dollar Bars): {precios_dollar_bars.shape}")

# (NOTA: Aún no calculamos retornos. Dejaremos precios_dollar_bars intacto para aplicarle FracDiff después)

Descargando datos de Yahoo Finance (Precios y Volumen)...


c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a f

Datos cronológicos originales (Time Bars): (16198, 23)
Nueva matriz tras compresión (Portfolio Dollar Bars): (2505, 23)


In [4]:
# =====================================================================
# 2. TRANSFORMACIÓN LÓPEZ DE PRADO: DIFERENCIACIÓN FRACCIONARIA (FFD)
# =====================================================================

def obtener_pesos_fracdiff(d, umbral=1e-4):
    """
    Calcula los pesos para la diferenciación fraccionaria.
    Los pesos disminuyen iterativamente. Se cortan cuando el valor
    absoluto del peso es menor que el 'umbral' (threshold).
    """
    w = [1.]
    k = 1
    while True:
        # Fórmula matemática recursiva de López de Prado para los pesos
        w_k = -w[-1] / k * (d - k + 1)
        if abs(w_k) < umbral:
            break
        w.append(w_k)
        k += 1
    
    # Invertimos los pesos para aplicarlos cronológicamente (el más reciente tiene peso 1)
    return np.array(w[::-1])

def aplicar_fracdiff_ffd(df_precios, d, umbral=1e-4):
    """
    Aplica FracDiff a un DataFrame entero usando una ventana fija (FFD).
    Esto evita la pérdida excesiva de datos al inicio de la serie.
    """
    print(f"Calculando Diferenciación Fraccionaria con d={d}...")
    
    # 1. Obtener los pesos matemáticos
    w = obtener_pesos_fracdiff(d, umbral)
    ventana = len(w)
    print(f"Tamaño de la ventana de memoria (días/barras retenidas): {ventana}")
    
    # Transformamos el DF a logaritmos para estabilizar la varianza antes de diferenciar
    df_log = np.log(df_precios)
    
    # 2. Crear un DataFrame vacío para los resultados
    df_diff = pd.DataFrame(index=df_log.index, columns=df_log.columns)
    
    # 3. Aplicar el producto punto (dot product) deslizando la ventana
    # Multiplicamos el vector de pesos (1D) por la matriz de precios de la ventana (2D)
    for i in range(ventana - 1, len(df_log)):
        # Tomamos el bloque temporal exacto de tamaño 'ventana' hasta el día 'i'
        corte = df_log.iloc[i - ventana + 1 : i + 1]
        
        # np.dot suma el peso histórico * precio histórico para los 23 activos a la vez
        df_diff.iloc[i] = np.dot(w, corte.values)
        
    # Eliminamos las primeras filas que no tenían historia suficiente para llenar la ventana
    return df_diff.dropna().astype(float)


# ---------------------------------------------------------------------
# EJECUCIÓN DEL ALGORITMO FRACDIFF
# ---------------------------------------------------------------------
# El parámetro 'd' (grado de diferenciación). 
# En el SP500, un valor entre 0.4 y 0.5 suele ser el punto dulce donde
# la serie se vuelve estacionaria (ADF test < 0.05) pero retiene >70% de memoria.
grado_d = 0.45 

# Reemplazamos tu antiguo: returns = np.log(precios_close).diff().dropna()
# Por la nueva matriz con memoria:
fracdiff_features = aplicar_fracdiff_ffd(precios_dollar_bars, d=grado_d)

print(f"Forma de los datos FracDiff finales: {fracdiff_features.shape}")

Calculando Diferenciación Fraccionaria con d=0.45...
Tamaño de la ventana de memoria (días/barras retenidas): 238
Forma de los datos FracDiff finales: (2268, 23)


In [8]:
from tensorflow.keras.utils import to_categorical

# =====================================================================
# 2. DEFINICIÓN DE ARQUITECTURA, TRIPLE BARRERA Y BASELINES
# =====================================================================

def create_triple_barrier_data(data_features, data_prices, input_window_size, output_window_size, pt_limit, sl_limit):
    """
    Sustituye a create_time_series_data.
    Genera etiquetas de clasificación usando el Método de la Triple Barrera.
    """
    X, y = [], []
    
    # Asegurarnos de que son arrays de numpy
    features_array = data_features.values if isinstance(data_features, pd.DataFrame) else data_features
    # Calculamos el precio promedio del portfolio para evaluar si "el mercado" toca la barrera
    precios_array = data_prices.mean(axis=1).values if isinstance(data_prices, pd.DataFrame) else data_prices

    for i in range(len(features_array) - input_window_size - output_window_size + 1):
        # 1. Ventana de Entrada (Input Features: FracDiff)
        input_sequence = features_array[i : i + input_window_size]
        X.append(input_sequence)
        
        # 2. Trayectoria Futura (Output: Precios reales para la barrera)
        p0 = precios_array[i + input_window_size - 1] # Precio en el momento t_0
        future_prices = precios_array[i + input_window_size : i + input_window_size + output_window_size]
        
        # Calcular los retornos acumulados de la trayectoria (Path)
        path_returns = (future_prices / p0) - 1
        
        # 3. Lógica de la Triple Barrera
        etiqueta = 1  # Por defecto, Clase 1 (Plano / Caduca el tiempo sin tocar barreras)
        
        for r in path_returns:
            if r >= pt_limit:     # Barrera Superior (Take Profit)
                etiqueta = 2      # Clase 2 (Sube)
                break
            elif r <= -sl_limit:  # Barrera Inferior (Stop Loss)
                etiqueta = 0      # Clase 0 (Baja)
                break
                
        y.append(etiqueta)

    X = np.array(X)
    # Convertimos a One-Hot Encoding para la capa Softmax de Keras
    y = to_categorical(y, num_classes=3) 
    
    return X, y


from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

def construir_modelo_mixto_prado(config, input_shape):
    """Construye un modelo Híbrido: CNN -> RNN -> Densa para CLASIFICACIÓN (Triple Barrera)"""
    model = Sequential()
    model.add(Input(shape=input_shape))
    
    # ---------------------------------------------------------
    # BLOQUE 1: EXTRACCIÓN DE PATRONES (CNN)
    # ---------------------------------------------------------
    model.add(Conv1D(filters=config['filtros_cnn'], 
                     kernel_size=config['kernel_size'], 
                     padding='same', 
                     activation='relu'))
    
    if input_shape[0] >= 10:
        model.add(MaxPooling1D(pool_size=2))
        
    # ---------------------------------------------------------
    # BLOQUE 2: MEMORIA TEMPORAL (RNN)
    # ---------------------------------------------------------
    model.add(config['tipo_rnn'](config['neuronas_rnn'], return_sequences=False))
    
    # ---------------------------------------------------------
    # BLOQUE 3: TOMA DE DECISIONES (Densas)
    # ---------------------------------------------------------
    model.add(Dropout(config['dropout']))
    
    if config['neuronas_densa'] > 0:
        model.add(Dense(config['neuronas_densa'], activation='relu'))
        
    # CAMBIO CRÍTICO PARA PRADO: 3 clases (Baja=0, Plano=1, Sube=2)
    model.add(Dense(3, activation='softmax')) 
    
    optimizador = Adam(learning_rate=config['lr'])
    model.compile(optimizer=optimizador, loss='categorical_crossentropy', metrics=['accuracy'])
    
    return model


def calcular_baselines_clasificacion(y_test, y_train):
    """
    Calcula la Precisión (Accuracy) para modelos base simples en clasificación.
    Los 'y' entran en One-Hot, sacamos la clase real con argmax.
    """
    y_test_classes = np.argmax(y_test, axis=1)
    y_train_classes = np.argmax(y_train, axis=1)
    
    # 1. Baseline "Always Buy": El mercado es alcista, siempre predecir Clase 2 (Sube)
    acc_always_buy = np.mean(y_test_classes == 2)
    
    # 2. Baseline "Always Flat": Siempre predecir Clase 1 (Plano)
    acc_always_flat = np.mean(y_test_classes == 1)
    
    # 3. Baseline "Estadístico Histórico" (Equivalente al Buy and Hold de Regresión):
    # Predecir siempre la clase que más apareció durante el periodo de Entrenamiento
    clase_mas_frecuente = np.bincount(y_train_classes).argmax()
    acc_most_frequent = np.mean(y_test_classes == clase_mas_frecuente)
    
    return acc_always_buy, acc_always_flat, acc_most_frequent


# Crear carpeta para guardar gráficas si no existe
os.makedirs('graficas_convergencia_tbm', exist_ok=True)

In [11]:
from tensorflow.keras.layers import LSTM, GRU

#  =====================================================================
# BANCOS DE PRUEBAS PARA REDES MIXTAS (CNN + RNN + Dense)
# =====================================================================
input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]


# ----------------- IN: 5 DÍAS (Memoria Corta) -----------------
# El escáner (Kernel) debe ser minúsculo (2) porque solo hay 5 días.
hp_in5_corto = [
    # 1. El Freno de Mano Total: 
    # Reducimos la GRU al mínimo absoluto (4 neuronas) y subimos el Dropout a 0.4.
    # Es casi imposible memorizar ruido con solo 4 neuronas y un 40% de amnesia.
    {'filtros_cnn': 8,  'kernel_size': 2, 'tipo_rnn': GRU,  'neuronas_rnn': 4,  'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005},
    
    # 2. El Aprendiz Extra-Lento: 
    # Mantenemos 16 filtros y 8 neuronas, pero bajamos el LR a 0.0001 (1e-4) y Drop a 0.3.
    # Obligamos a la red a dar pasos milimétricos. Evitaremos el "efecto gancho" seguro.
    {'filtros_cnn': 16, 'kernel_size': 2, 'tipo_rnn': LSTM, 'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.3, 'lr': 0.0001},
    
    # 3. El Filtro Extremo: 
    # Dropout al 50%. A cada paso, la red olvida la mitad de lo que ha visto.
    # Solo pasará la señal matemática si es escandalosamente obvia.
    {'filtros_cnn': 8,  'kernel_size': 2, 'tipo_rnn': LSTM, 'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0005}
]

hp_in5_largo = [
    # 1. El Freno Extremo:
    # Quitamos la capa Densa (es un nido de overfitting para tan pocos datos).
    # Reducimos filtros y neuronas al mínimo. Dropout altísimo (0.4).
    {'filtros_cnn': 8,  'kernel_size': 2, 'tipo_rnn': GRU,  'neuronas_rnn': 4,  'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005},
    
    # 2. El Modelo Amnésico y Lento:
    # LSTM muy pequeña, kernel un poco más amplio (3 días), sin densa.
    # Dropout del 50% y un Learning Rate minúsculo. 
    # Obligamos a la red a no creerse nada de lo que ve a menos que sea una tendencia brutal.
    {'filtros_cnn': 8, 'kernel_size': 3, 'tipo_rnn': LSTM, 'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]


# ----------------- IN: 10 DÍAS (Memoria Quincenal) -----------------
# Aquí ya entra el MaxPooling que corta la secuencia a la mitad.
hp_in10_corto = [
    # 1. El "Ultra-Lento":
    # Bajamos los filtros y las neuronas a la mitad (8). Learning Rate bajísimo (1e-4).
    # Le costará horrores aprender, lo que evitará que la validación se dispare de golpe.
    {'filtros_cnn': 8,  'kernel_size': 3, 'tipo_rnn': GRU,  'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0001},

    # 2. El "Amnésico":
    # Mantenemos un poco de capacidad (16 filtros/neuronas) pero con un Dropout brutal (50%).
    # Olvidando la mitad de la información a cada paso, solo aprenderá tendencias reales.
    {'filtros_cnn': 16, 'kernel_size': 3, 'tipo_rnn': LSTM, 'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0005},

    # 3. El Escáner Semanal:
    # Aumentamos el Kernel a 5. La CNN mirará los 5 días de la semana de golpe 
    # en lugar de mirar de 3 en 3 días. Todo con máxima restricción.
    {'filtros_cnn': 8,  'kernel_size': 5, 'tipo_rnn': GRU,  'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005}
]

hp_in10_largo = [
    # 1. El Freno Extremo (Visión Semanal):
    # Mantenemos el escáner de 5 días, pero reducimos drásticamente los filtros 
    # y la memoria (8). Fuera la capa densa. Subimos el Dropout a 0.4.
    {'filtros_cnn': 8,  'kernel_size': 5, 'tipo_rnn': LSTM, 'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005},
    
    # 2. El Modelo Amnésico y Lento:
    # Le damos un poco más de filtros (16) para que busque más patrones, pero 
    # lo castigamos con un Dropout del 50% y un Learning Rate microscópico (1e-4).
    # O encuentra una tendencia brutalmente clara, o no aprenderá nada.
    {'filtros_cnn': 16, 'kernel_size': 5, 'tipo_rnn': GRU,  'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]

# ----------------- IN: 30 DÍAS (Memoria Mensual) -----------------
# Mucho ruido. Necesitamos un embudo estricto (Dropout alto).
hp_in30_corto = [
    # 1. El Escáner Conservador:
    # Kernel 5 (ve semanas). Pocos filtros (8) y memoria ligera (GRU 16).
    # Bajamos el LR a 0.0005 y subimos el Dropout a 0.4 para frenar el gancho suave.
    {'filtros_cnn': 8,  'kernel_size': 5, 'tipo_rnn': GRU,  'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005},
    
    # 2. El Analista Lento:
    # Más filtros (16) y Kernel de 3 días. 
    # LR microscópico (1e-4) y Dropout altísimo (0.5). Obligamos a la LSTM a 
    # no precipitarse al predecir el día de mañana.
    {'filtros_cnn': 16, 'kernel_size': 3, 'tipo_rnn': LSTM, 'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]

hp_in30_largo = [
    # 1. El Analista Macro (Moderado):
    # Mantenemos el escáner de semanas enteras (Kernel 5) pero bajamos la memoria a 16.
    # Fulminamos la capa Densa. Subimos el Dropout al 40%.
    {'filtros_cnn': 16, 'kernel_size': 5, 'tipo_rnn': GRU,  'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.4, 'lr': 0.0005},
    
    # 2. El Escéptico a Largo Plazo (Extremo):
    # Un poco más de filtros (32) para ver más variables, pero lo frenamos con 
    # un Dropout bestial del 50% y un LR muy bajo. 
    # O encuentra algo obvio, o se quedará plano (lo cual es mejor que sobreajustar).
    {'filtros_cnn': 32, 'kernel_size': 5, 'tipo_rnn': LSTM, 'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]

# ----------------- IN: 90 DÍAS (Memoria Trimestral) -----------------
# Riesgo inmenso de sobreajuste. 
hp_in90_corto = [
    # 1. El Compresor Extremo (Lento y seguro):
    # Mantenemos el Kernel de 10 (la CNN lee quincenas enteras, no días).
    # Bajamos los filtros a 8 y la GRU a 8. 
    # El LR baja a 0.0005 y el Dropout sube al 50%.
    # Queremos que la red ignore el 90% de la información irrelevante.
    {'filtros_cnn': 8, 'kernel_size': 10, 'tipo_rnn': GRU,  'neuronas_rnn': 8,  'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0005},
    
    # 2. El Escáner Semanal Asfixiado:
    # Kernel de 5 (ve semanas). 16 filtros y LSTM de 16.
    # Pero le metemos el freno de mano total: LR de 0.0001 (muy lento) y Dropout de 0.5.
    # Impedirá que la validación salga disparada en las primeras 10 épocas.
    {'filtros_cnn': 16, 'kernel_size': 5,  'tipo_rnn': LSTM, 'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]

hp_in90_largo = [
    # 1. El Macro-Analista Estricto:
    # Mantenemos el Kernel de 10 (la CNN lee los datos en bloques de dos semanas).
    # Bajamos los filtros y la memoria a 16. ¡Fuera la capa Densa!
    # Dropout máximo (0.5) para que no se memorice el ruido diario.
    {'filtros_cnn': 16, 'kernel_size': 10, 'tipo_rnn': GRU,  'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0005},
    
    # 2. El Observador Trimestral Lento:
    # 32 filtros (lo máximo absoluto que le vamos a permitir) y Kernel 5.
    # LSTM de 16 neuronas sin capa densa. 
    # LR bajísimo (0.0001) para que los ajustes sean milimétricos.
    {'filtros_cnn': 32, 'kernel_size': 5,  'tipo_rnn': LSTM, 'neuronas_rnn': 16, 'neuronas_densa': 0, 'dropout': 0.5, 'lr': 0.0001}
]

lista_hiperparametros = []

# Matrices para reportar resultados finales de las Redes Recurrentes
matriz_mae_train_rnn = np.zeros((4, 4))
matriz_mae_val_rnn = np.zeros((4, 4))
matriz_mae_rnn = np.zeros((4, 4)) # Esta es la de Test que ya tenías

matriz_mae_naive = np.zeros((4, 4))
matriz_mae_sma = np.zeros((4, 4))
matriz_mae_bh = np.zeros((4, 4))

# Aumentamos la paciencia a 15 épocas
early_stop = EarlyStopping(
    monitor ='val_loss', 
    patience = 20,               # <--- CAMBIO AQUÍ
    restore_best_weights = True  # IMPORTANTE: Que devuelva los pesos de la mejor época
)


In [12]:
# =====================================================================
# INICIALIZACIÓN DE MATRICES (Asegúrate de ejecutar esto antes del bucle)
# =====================================================================
matriz_acc_naive_val = np.zeros((len(input_windows), len(output_windows)))
matriz_acc_most_frequent_val = np.zeros((len(input_windows), len(output_windows)))
matriz_acc_naive_test = np.zeros((len(input_windows), len(output_windows)))
matriz_acc_most_frequent_test = np.zeros((len(input_windows), len(output_windows)))

matriz_acc_train_mixtas = np.zeros((len(input_windows), len(output_windows)))
matriz_acc_val_mixtas = np.zeros((len(input_windows), len(output_windows)))
matriz_acc_test_mixtas = np.zeros((len(input_windows), len(output_windows)))

# =====================================================================
# 4. BUCLE PRINCIPAL: REDES MIXTAS + LÓPEZ DE PRADO
# =====================================================================

print("\nIniciando entrenamiento de modelos MIXTOS de Clasificación (Triple Barrera)...")

for i, in_w in enumerate(input_windows):
    for j, out_w in enumerate(output_windows):
        
        # Barreras Dinámicas
        pt_dinamico = 0.01 + (0.001 * out_w)
        sl_dinamico = 0.01 + (0.001 * out_w)

        print(f"\n=======================================================")
        print(f" Ventana Entrada (Memoria): {in_w} | Salida (Horizonte): {out_w}")
        print(f"=======================================================")
        
        # 1. Crear datos con FracDiff y Triple Barrera
        X, y = create_triple_barrier_data(
            fracdiff_features, 
            precios_dollar_bars, 
            in_w, 
            out_w, 
            pt_limit=pt_dinamico, 
            sl_limit=sl_dinamico
        )
        
        # 2. Separación CRONOLÓGICA
        split_1 = int(len(X) * 0.70)
        split_2 = int(len(X) * 0.90)
        
        X_train, y_train = X[:split_1], y[:split_1]
        X_val, y_val = X[split_1:split_2], y[split_1:split_2]
        X_test, y_test = X[split_2:], y[split_2:]
        
        # =====================================================================
        # 3. Baselines de Clasificación
        # =====================================================================
        acc_buy_val, acc_flat_val, acc_freq_val = calcular_baselines_clasificacion(y_val, y_train)
        matriz_acc_naive_val[i, j] = acc_buy_val
        matriz_acc_most_frequent_val[i, j] = acc_freq_val
        
        acc_buy_test, acc_flat_test, acc_freq_test = calcular_baselines_clasificacion(y_test, y_train)
        matriz_acc_naive_test[i, j] = acc_buy_test
        matriz_acc_most_frequent_test[i, j] = acc_freq_test

        print("--- Baselines (Accuracy) VALIDACIÓN ---")
        print(f"Always Buy: {acc_buy_val:.4f} | Most Frequent: {acc_freq_val:.4f}")
        print("--- Baselines (Accuracy) TEST ---")
        print(f"Always Buy: {acc_buy_test:.4f} | Most Frequent: {acc_freq_test:.4f}\n")

        # 4. Búsqueda del mejor modelo Mixto
        mejor_val_loss = float('inf')
        mejor_modelo = None
        mejor_historial = None
        mejor_config = None

        # Selección de hiperparámetros
        if in_w == 5:
            lista_a_probar = hp_in5_corto if out_w in [1, 5] else hp_in5_largo
            nombre_lista = "Mixto In:5 Corto" if out_w in [1, 5] else "Mixto In:5 Largo"
        elif in_w == 10:
            lista_a_probar = hp_in10_corto if out_w in [1, 5] else hp_in10_largo
            nombre_lista = "Mixto In:10 Corto" if out_w in [1, 5] else "Mixto In:10 Largo"
        elif in_w == 30:
            lista_a_probar = hp_in30_corto if out_w in [1, 5] else hp_in30_largo
            nombre_lista = "Mixto In:30 Corto" if out_w in [1, 5] else "Mixto In:30 Largo"
        elif in_w == 90:
            lista_a_probar = hp_in90_corto if out_w in [1, 5] else hp_in90_largo
            nombre_lista = "Mixto In:90 Corto" if out_w in [1, 5] else "Mixto In:90 Largo"

        print(f" -> Usando banco de pruebas: [{nombre_lista}]")
        
        # Opcional: Calcular pesos de clases para combatir el desbalanceo
        from sklearn.utils.class_weight import compute_class_weight
        y_train_enteros = np.argmax(y_train, axis=1)
        clases_unicas = np.unique(y_train_enteros)
        pesos = compute_class_weight(class_weight='balanced', classes=clases_unicas, y=y_train_enteros)
        diccionario_pesos = dict(zip(clases_unicas, pesos))
        
        for config in lista_a_probar:
            nombre_rnn = config['tipo_rnn'].__name__
            print(f" -> Entrenando: CNN({config['filtros_cnn']}F, K{config['kernel_size']}) + {nombre_rnn}({config['neuronas_rnn']}) + Densa({config['neuronas_densa']})")
            
            # Usamos la nueva constructora mixta para Prado
            modelo = construir_modelo_mixto_prado(config, input_shape=(in_w, 23))
            
            historial = modelo.fit(X_train, y_train, 
                                   validation_data=(X_val, y_val),
                                   epochs=50, 
                                   batch_size=64, 
                                   callbacks=[early_stop],
                                   class_weight=diccionario_pesos, # Ayuda a que no apueste siempre "Sube"
                                   verbose=0)
            
            val_loss_actual = min(historial.history['val_loss'])
            
            if val_loss_actual < mejor_val_loss:
                mejor_val_loss = val_loss_actual
                mejor_modelo = modelo
                mejor_historial = historial
                mejor_config = config
        
        print(f"\n[GANADOR MIXTO] CNN({mejor_config['filtros_cnn']}F) + {mejor_config['tipo_rnn'].__name__}({mejor_config['neuronas_rnn']})")
        
        # 5. Evaluación final del GANADOR
        loss_train, acc_train_ganador = mejor_modelo.evaluate(X_train, y_train, verbose=0)
        loss_val, acc_val_ganador = mejor_modelo.evaluate(X_val, y_val, verbose=0)
        loss_test, acc_test_ganador = mejor_modelo.evaluate(X_test, y_test, verbose=0)
        
        matriz_acc_train_mixtas[i, j] = acc_train_ganador
        matriz_acc_val_mixtas[i, j] = acc_val_ganador
        matriz_acc_test_mixtas[i, j] = acc_test_ganador
        
        print(f"Accuracy en TRAIN:      {acc_train_ganador:.4f}")
        print(f"Accuracy en VALIDACIÓN: {acc_val_ganador:.4f}")
        print(f"Accuracy en TEST:       {acc_test_ganador:.4f}")
        
        # 6. Guardar Gráficas de Convergencia (Clasificación)
        import matplotlib.pyplot as plt
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Variables para el título
        f_cnn = mejor_config['filtros_cnn']
        k_cnn = mejor_config['kernel_size']
        nom_rnn = mejor_config['tipo_rnn'].__name__
        n_rnn = mejor_config['neuronas_rnn']
        n_den = mejor_config['neuronas_densa']
        l_rate = mejor_config['lr']
        d_out = mejor_config['dropout']
        
        titulo_arqui = f"CNN({f_cnn}F, K{k_cnn}) + {nom_rnn}({n_rnn}) + Densa({n_den})"
        fig.suptitle(f"Clasificación Mixta: {titulo_arqui}\nLR: {l_rate} | Drop: {d_out} | (In:{in_w} - Out:{out_w})")
        
        # Gráfica de Loss
        ax1.plot(mejor_historial.history['loss'], label='Loss Entrenamiento')
        ax1.plot(mejor_historial.history['val_loss'], label='Loss Validación')
        ax1.set_xlabel('Épocas')
        ax1.set_ylabel('Categorical Crossentropy')
        ax1.legend()
        ax1.grid(True)
        
        # Gráfica de Accuracy
        ax2.plot(mejor_historial.history['accuracy'], label='Accuracy Entrenamiento')
        ax2.plot(mejor_historial.history['val_accuracy'], label='Accuracy Validación')
        ax2.set_xlabel('Épocas')
        ax2.set_ylabel('Accuracy')
        ax2.legend()
        ax2.grid(True)
        
        # Guardar la imagen
        os.makedirs('graficas_convergencia_mixtas_prado', exist_ok=True)
        nombre_archivo = f"graficas_convergencia_mixtas_prado/conver_in{in_w}_out{out_w}.png"
        plt.savefig(nombre_archivo)
        plt.close()


Iniciando entrenamiento de modelos MIXTOS de Clasificación (Triple Barrera)...

 Ventana Entrada (Memoria): 5 | Salida (Horizonte): 1
--- Baselines (Accuracy) VALIDACIÓN ---
Always Buy: 0.2456 | Most Frequent: 0.5553
--- Baselines (Accuracy) TEST ---
Always Buy: 0.1454 | Most Frequent: 0.7753

 -> Usando banco de pruebas: [Mixto In:5 Corto]
 -> Entrenando: CNN(8F, K2) + GRU(4) + Densa(0)
 -> Entrenando: CNN(16F, K2) + LSTM(8) + Densa(0)
 -> Entrenando: CNN(8F, K2) + LSTM(8) + Densa(0)

[GANADOR MIXTO] CNN(8F) + GRU(4)
Accuracy en TRAIN:      0.4968
Accuracy en VALIDACIÓN: 0.5509
Accuracy en TEST:       0.7709

 Ventana Entrada (Memoria): 5 | Salida (Horizonte): 5
--- Baselines (Accuracy) VALIDACIÓN ---
Always Buy: 0.4469 | Most Frequent: 0.4469
--- Baselines (Accuracy) TEST ---
Always Buy: 0.4690 | Most Frequent: 0.4690

 -> Usando banco de pruebas: [Mixto In:5 Corto]
 -> Entrenando: CNN(8F, K2) + GRU(4) + Densa(0)
 -> Entrenando: CNN(16F, K2) + LSTM(8) + Densa(0)
 -> Entrenando: CNN(

In [ ]:
# =====================================================================
# 5. RESULTADOS FINALES EN CLASIFICACIÓN (Tablas para tu GitHub y Presentación)
# =====================================================================

print("\n\n" + "="*60)
print("MATRIZ DE RESULTADOS FINALES EN ENTRENAMIENTO (ACCURACY)")
print("="*60)
df_rnn_train_acc = pd.DataFrame(matriz_acc_train_rnn, 
                            index=[f'In_{w}' for w in input_windows], 
                            columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_train_acc)

print("\n" + "="*60)
print("MATRIZ DE RESULTADOS FINALES EN VALIDACIÓN (ACCURACY)")
print("="*60)
df_rnn_val_acc = pd.DataFrame(matriz_acc_val_rnn, 
                          index=[f'In_{w}' for w in input_windows], 
                          columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_val_acc)

print("\n" + "="*60)
print("MATRIZ DE RESULTADOS FINALES EN TEST (ACCURACY)")
print("="*60)
df_rnn_test_acc = pd.DataFrame(matriz_acc_test_rnn, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_test_acc)

print("\n" + "="*60)
print("MATRIZ BASELINE 'ALWAYS BUY' (VALIDACIÓN)")
print("="*60)
# Usamos la matriz donde guardamos el accuracy de "Always Buy"
df_always_buy_val = pd.DataFrame(matriz_acc_naive_val, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_always_buy_val)

print("\n" + "="*60)
print("MATRIZ BASELINE 'ALWAYS BUY' (TEST)")
print("="*60)
df_always_buy_test = pd.DataFrame(matriz_acc_naive_test, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_always_buy_test)

print("\n" + "="*60)
print("MATRIZ BASELINE 'MOST FREQUENT' (VALIDACIÓN) - Eq. Buy&Hold")
print("="*60)
df_most_freq_val = pd.DataFrame(matriz_acc_most_frequent_val, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_most_freq_val)

print("\n" + "="*60)
print("MATRIZ BASELINE 'MOST FREQUENT' (TEST) - Eq. Buy&Hold")
print("="*60)
df_most_freq_test = pd.DataFrame(matriz_acc_most_frequent_test, 
                     index=[f'In_{w}' for w in input_windows], 
                     columns=[f'Out_{w}' for w in output_windows])
print(df_most_freq_test)